In [0]:
from pyspark.sql import functions as F
source_path = "/Volumes/workspace/default/my_volume/customers-100.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(source_path)

display(df)
for col_name in df.columns:
    df = df.withColumnRenamed(
        col_name,
        col_name.strip().replace(" ", "_")
    )

print(df.columns)
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_customer")
bronze_df = spark.table("workspace.default.bronze_customer")
display(bronze_df)

silver_df = bronze_df.dropDuplicates()

display(silver_df)

silver_df = silver_df.filter(
    F.col("Customer_Id").isNotNull()
)

display(silver_df)
silver_df = silver_df \
    .withColumn("First_Name", F.trim(F.col("First_Name"))) \
    .withColumn("Last_Name", F.trim(F.col("Last_Name"))) \
    .withColumn("City", F.trim(F.col("City"))) \
    .withColumn("Country", F.trim(F.col("Country"))) \
    .withColumn("Email", F.trim(F.col("Email")))

display(silver_df)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_customer")
silver_df = spark.table("workspace.default.silver_customer")

display(silver_df)
gold_df = silver_df.groupBy("Country") \
    .agg(
        F.count("*").alias("customer_count")
    ) \
    .orderBy(F.desc("customer_count"))

display(gold_df)
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_customer_country3")
